# Corrected-teacher eval: general knowledge

Part of plan.md step 3.2 -- same suite as 1.2's baseline eval
(`02c_baseline_general_knowledge.ipynb`): MMLU (5-shot), ARC-Challenge
(25-shot), Winogrande (5-shot), full test sets, shot counts pinned in
`bonsai.eval.general_knowledge.NUM_FEWSHOT`. Only the model pointer changes,
to the corrected teacher (`sartajbhuvaji/bonsai-qwen3-30b-a3b-teacher-corrected`, 3.1). Tracked, not
optimized-for -- correction targets TypeScript specifically, so this is a
check against regressing general capability, not a target to improve.

In [1]:
import json

from bonsai.config import load_config
from bonsai.eval.general_knowledge import run_general_knowledge
from bonsai.tracking import log_metric, new_run, save_config_snapshot

config = load_config("../configs/qwen3-30b-a3b.toml")
model_repo_id = "sartajbhuvaji/bonsai-qwen3-30b-a3b-teacher-corrected"
run_dir = new_run("teacher-corrected-eval")
save_config_snapshot(run_dir, config)

print(f"model: {model_repo_id}")
print(f"run dir: {run_dir}")

model: sartajbhuvaji/bonsai-qwen3-30b-a3b-teacher-corrected
run dir: runs/20260921T192638Z-a6a8d8-teacher-corrected-eval


In [2]:
general_knowledge = run_general_knowledge(
    model_repo_id, revision="main", dtype="bfloat16", batch_size="auto"
)

for task, result in general_knowledge.items():
    print(task, result["results"][task])

Loading weights:   0%|          | 0/531 [00:00<?, ?it/s]

README.md:   0%|          | 0.00/53.2k [00:00<?, ?B/s]

dataset_infos.json:   0%|          | 0.00/138k [00:00<?, ?B/s]

abstract_algebra/test-00000-of-00001.par(…): reconstructing file:   0%|          |  0.00B / 9.96kB            

abstract_algebra/test-00000-of-00001.par(…): downloading bytes:           |  0.00B            

abstract_algebra/validation-00000-of-000(…): reconstructing file:   0%|          |  0.00B / 3.73kB            

... (5,253 outputs left out)

Running loglikelihood requests:  96%|█████████▌| 2433/2534 [00:27<00:00, 116.84it/s]

Running loglikelihood requests:  99%|█████████▊| 2497/2534 [00:28<00:00, 131.06it/s]

Running loglikelihood requests: 100%|██████████| 2534/2534 [00:28<00:00, 90.24it/s] 

mmlu {'alias': 'mmlu', 'name': 'mmlu', 'sample_len': 14042, 'acc,none': 0.8082182025352513, 'acc_stderr,none': 0.0031548993511966634, 'sample_count': {'acc,none': 14042}}
arc_challenge {'name': 'arc_challenge', 'alias': 'arc_challenge', 'sample_len': 1172, 'acc,none': 0.643344709897611, 'acc_stderr,none': 0.013998056902620168, 'acc_norm,none': 0.6689419795221843, 'acc_norm_stderr,none': 0.013752062419817994}
winogrande {'name': 'winogrande', 'alias': 'winogrande', 'sample_len': 1267, 'acc,none': 0.7805840568271507, 'acc_stderr,none': 0.011631268360607785}


In [3]:
(run_dir / "general-knowledge.json").write_text(
    json.dumps(general_knowledge, indent=2, default=str)
)

for task, result in general_knowledge.items():
    log_metric(run_dir, task=task, **result["results"][task])